# 02 — Text and Tokenization

## Goal

Language models do not operate directly on strings.

Before text can enter a neural network, it must be converted into a
sequence of discrete symbols and then into integer token IDs.

This lesson builds a minimal character-level tokenizer from scratch in
order to understand:

- what a token is,
- how a vocabulary is constructed,
- how tokens are mapped to integer IDs,
- how encoding and decoding work,
- how unknown tokens should be handled,
- and what information can be lost during tokenization.

The goal is conceptual clarity rather than tokenizer efficiency.

## Scope

We will first implement a deterministic character-level tokenizer using
plain Python.

For now, one Python character will correspond to one token.

Later, we will examine why modern language models usually use subword
tokenization instead of character-level tokenization.

## 1. Raw Text and Characters

A Python string is already an ordered sequence.

For a character-level tokenizer, we treat each character as one token.

Whitespace and newline characters are also part of the text and should
not be silently removed.

In [1]:
text: str = "hello world\nhello llm"
print(text)
print(repr(text))
print("number of characters:", len(text))


hello world
hello llm
'hello world\nhello llm'
number of characters: 21


## 2. Building a Vocabulary

A vocabulary is the set of token types that the tokenizer knows about.

For a character-level tokenizer, the vocabulary contains the unique
characters observed in the training text.

The vocabulary must have a deterministic ordering because token IDs
depend on that ordering.

Using a `set` removes duplicates, but sets do not define the token-ID
assignment we want. We therefore sort the unique characters before
assigning IDs.

In [2]:
vocab: list[str] = sorted(set(text))
print(vocab)
print("vocabulary size: ", len(vocab))

['\n', ' ', 'd', 'e', 'h', 'l', 'm', 'o', 'r', 'w']
vocabulary size:  10


## 3. Mapping Tokens to Integer IDs

Neural-network layers operate on numerical tensors rather than Python
strings.

We therefore assign every token in the vocabulary a unique integer ID.

We maintain mappings in both directions:

<pre>
token  → token ID
token ID → token
</pre>

The first mapping is used during encoding.

The second mapping is used during decoding.

In [3]:
char_to_id: dict[str, int] = {char: index for index, char in enumerate(vocab)}

id_to_char: dict[int, str] = {index: char for index, char in enumerate(vocab)}

print(char_to_id)
print(id_to_char)

{'\n': 0, ' ': 1, 'd': 2, 'e': 3, 'h': 4, 'l': 5, 'm': 6, 'o': 7, 'r': 8, 'w': 9}
{0: '\n', 1: ' ', 2: 'd', 3: 'e', 4: 'h', 5: 'l', 6: 'm', 7: 'o', 8: 'r', 9: 'w'}


## 4. Encoding and Decoding

Encoding converts a sequence of tokens into integer token IDs.

For a character-level tokenizer,

$$
[c_1, c_2, \ldots, c_T]
\longrightarrow
[i_1, i_2, \ldots, i_T].
$$

Decoding performs the inverse mapping:

$$
[i_1, i_2, \ldots, i_T]
\longrightarrow
[c_1, c_2, \ldots, c_T].
$$

For text that is fully covered by the vocabulary, a correct tokenizer
should satisfy the round-trip property

$$
\operatorname{decode}(\operatorname{encode}(x)) = x.
$$

The token IDs are categorical identifiers. Their numerical values do
not encode semantic similarity or ordering.

In [6]:
from collections.abc import Mapping, Sequence


def encode(text: str, token_to_id: Mapping[str, int]) -> list[int]:
    return [token_to_id[char] for char in text]


Decoding reverses the mapping and concatenates the recovered character
tokens into a Python string.

In [7]:
def decode(token_ids: Sequence[int], id_to_token: Mapping[int, str]) -> str:
    return "".join(id_to_token[token_id] for token_id in token_ids)

In [8]:
# test
encoded: list[int] = encode(
    text,
    char_to_id,
)

decoded: str = decode(
    encoded,
    id_to_char,
)

print("original:")
print(repr(text))

print("\nencoded:")
print(encoded)

print("\ndecoded:")
print(repr(decoded))

print("\nround trip:")
print(decoded == text)

original:
'hello world\nhello llm'

encoded:
[4, 3, 5, 5, 7, 1, 9, 7, 8, 5, 2, 0, 4, 3, 5, 5, 7, 1, 5, 5, 6]

decoded:
'hello world\nhello llm'

round trip:
True


## 5. Unknown Tokens

The vocabulary constructed from a finite corpus cannot necessarily
represent every possible future input.

If encoding encounters a token that is absent from the vocabulary, the
simplest dictionary lookup fails.

This forces us to define an explicit policy for unknown input rather
than silently changing or discarding it.

In [9]:
new_text: str = "hello!"

encode(new_text, char_to_id)

KeyError: '!'

In [10]:
def encode_strict(text: str, token_to_id: Mapping[str, int]) -> list[int]:
    token_ids: list[int] = []

    for char in text:
        if char not in token_to_id:
            raise ValueError(
                f"Unknown token: {char!r}"
            )  # !r use `repr()` to show, so "\n" won't be a newline
        token_ids.append(token_to_id[char])

    return token_ids


assert (
    decode(
        encode_strict(text, char_to_id),
        id_to_char,
    )
    == text
)

try:
    encode_strict(
        "hello!",
        char_to_id,
    )
except ValueError as error:
    print(error)

Unknown token: '!'


### Encoding Takeaway

Our first tokenizer is intentionally strict:

- every input character must exist in the vocabulary,
- encoding does not silently modify the input,
- decoding exactly reconstructs supported text,
- unsupported characters produce an explicit error.

This makes the tokenizer easy to reason about and preserves the
round-trip property for all supported inputs.

A later tokenizer may introduce special tokens such as `<UNK>`, but
doing so changes the information-preservation behavior and should be a
deliberate design decision.

## 6. Why Character-Level Tokenization Is Limited

A character-level tokenizer has several useful properties:

- the vocabulary is small,
- the implementation is simple,
- arbitrary words can be represented as long as their characters are known.

However, it also produces relatively long token sequences.

For example,

<pre>
transformer
↓
t r a n s f o r m e r
</pre>

contains 11 character tokens.

A language model must perform computation over every token position, so
longer token sequences increase the amount of work required by the
model.

Character tokens also ignore recurring multi-character patterns such
as

<pre>
ing
tion
transform
er
</pre>

that may occur frequently in a corpus.

A subword tokenizer attempts to learn useful multi-character units from
data.

The goal is not necessarily to recover complete words. Instead, it
constructs a vocabulary of reusable symbol sequences that can represent
text with fewer tokens than pure character tokenization.

## 7. Byte-Pair Encoding: Core Idea

We begin with a vocabulary containing individual symbols.

For a word such as

<pre>
lower
</pre>

the initial representation is

<pre>
l  o  w  e  r
</pre>

BPE repeatedly performs three steps:

1. count adjacent symbol pairs,
2. find the most frequent pair,
3. merge that pair into a new symbol.

For example, if

<pre>
l o
</pre>

is the most frequent pair, it may be merged into

<pre>
lo
</pre>

so that

<pre>
l  o  w  e  r
</pre>

becomes

<pre>
lo  w  e  r
</pre>

The process is repeated, gradually expanding the vocabulary from
characters toward frequently occurring subword units.

In [11]:
corpus_words: list[str] = [
    "low",
    "lower",
    "lowest",
    "low",
    "lower",
    "newer",
    "wider",
]

symbol_sequences: list[list[str]] = [list(word) for word in corpus_words]

for sequence in symbol_sequences:
    print(sequence)

['l', 'o', 'w']
['l', 'o', 'w', 'e', 'r']
['l', 'o', 'w', 'e', 's', 't']
['l', 'o', 'w']
['l', 'o', 'w', 'e', 'r']
['n', 'e', 'w', 'e', 'r']
['w', 'i', 'd', 'e', 'r']


## 8. Counting Adjacent Symbol Pairs

The first step of BPE is to count how often every adjacent pair appears.

For the sequence

<pre>
l  o  w  e  r
</pre>

the adjacent pairs are

<pre>
(l, o)
(o, w)
(w, e)
(e, r)
</pre>

For a sequence of length $T$, there are

$$
T - 1
$$

adjacent pairs.

Pair frequencies are accumulated over the entire training corpus.

In [12]:
from collections.abc import Sequence

Pair = tuple[str, str]


def count_adjacent_pairs(sequences: Sequence[Sequence[str]]) -> dict[Pair, int]:
    counts: dict[Pair, int] = {}

    for sequence in sequences:
        for index in range(len(sequence) - 1):
            pair: Pair = (sequence[index], sequence[index + 1])
            counts[pair] = counts.get(pair, 0) + 1

    return counts


pair_counts: dict[Pair, int] = count_adjacent_pairs(symbol_sequences)

for pair, count in pair_counts.items():
    print(pair, count)

('l', 'o') 5
('o', 'w') 5
('w', 'e') 4
('e', 'r') 4
('e', 's') 1
('s', 't') 1
('n', 'e') 1
('e', 'w') 1
('w', 'i') 1
('i', 'd') 1
('d', 'e') 1


In [13]:
sorted_pairs: list[tuple[Pair, int]] = sorted(
    pair_counts.items(),
    key=lambda item: item[1],
    reverse=True,
)

for pair, count in sorted_pairs:
    print(pair, count)

('l', 'o') 5
('o', 'w') 5
('w', 'e') 4
('e', 'r') 4
('e', 's') 1
('s', 't') 1
('n', 'e') 1
('e', 'w') 1
('w', 'i') 1
('i', 'd') 1
('d', 'e') 1


In [14]:
def most_frequent_pair(pair_counts: dict[Pair, int]) -> Pair:
    if not pair_counts:
        raise ValueError("Cannot select from empty pair counts")

    return max(pair_counts, key=pair_counts.get)


best_pair: Pair = most_frequent_pair(pair_counts)

print("most frequent pair:", best_pair)
print("frequency:", pair_counts[best_pair])

most frequent pair: ('l', 'o')
frequency: 5


### Pairs of Symbols, Not Necessarily Characters

BPE always merges two adjacent **symbols**, but a symbol may represent
more than one character.

At the beginning, every symbol may be a single character:

<pre>
l | o | w
</pre>

After merging `(l, o)`, the sequence becomes:

<pre>
lo | w
</pre>

The next pair `(lo, w)` still contains two symbols, even though it spans
three characters.

Repeated binary merges are therefore sufficient to construct tokens of
arbitrary length.

## 9. Merging a Symbol Pair

Once a symbol pair has been selected, every non-overlapping occurrence
of that adjacent pair is replaced by a new merged symbol.

For example, if the selected pair is

$$
(\text{l}, \text{o}),
$$

then

<pre>
l | o | w | e | r
</pre>

becomes

<pre>
lo | w | e | r
</pre>

The merged symbol `lo` is now treated as a single symbol in future BPE
iterations.

Importantly, BPE merges **symbols**, not necessarily individual
characters. A later merge may therefore combine symbols such as

<pre>
lo + w → low
</pre>

or

<pre>
low + er → lower
</pre>

Merging is performed from left to right and occurrences must not
overlap.

In [ ]:
def merge_pair(sequence: Sequence[str], pair: Pair) -> list[str]:
    merged: list[str] = []
    index: int = 0

    while index < len(sequence):
        if (
            index < len(sequence) - 1
            and sequence[index] == pair[0]
            and sequence[index + 1] == pair[1]
        ):
            merged.append(pair[0] + pair[1])
            index += 2
        else:
            merged.append(sequence[index])
            index += 1

    return merged


# test
sequence: list[str] = ["l", "o", "w", "e", "r"]

result: list[str] = merge_pair(
    sequence,
    ("l", "o"),
)

print(result)

result = merge_pair(
    result,
    ("lo", "w"),
)

print(result)

sequence = ["a", "a", "a"]

result = merge_pair(
    sequence,
    ("a", "a"),
)

print(result)

['lo', 'w', 'e', 'r']
['low', 'e', 'r']
['aa', 'a']


In [18]:
def merge_corpus(
    sequences: Sequence[Sequence[str]],
    pair: Pair,
) -> list[list[str]]:
    return [merge_pair(sequence, pair) for sequence in sequences]

In [21]:
best_pair: Pair = most_frequent_pair(pair_counts)

merged_sequences: list[list[str]] = merge_corpus(symbol_sequences, best_pair)

print("selected pair: ", best_pair)

for before, after in zip(symbol_sequences, merged_sequences, strict=True):
    print(before, "->", after)

selected pair:  ('l', 'o')
['l', 'o', 'w'] -> ['lo', 'w']
['l', 'o', 'w', 'e', 'r'] -> ['lo', 'w', 'e', 'r']
['l', 'o', 'w', 'e', 's', 't'] -> ['lo', 'w', 'e', 's', 't']
['l', 'o', 'w'] -> ['lo', 'w']
['l', 'o', 'w', 'e', 'r'] -> ['lo', 'w', 'e', 'r']
['n', 'e', 'w', 'e', 'r'] -> ['n', 'e', 'w', 'e', 'r']
['w', 'i', 'd', 'e', 'r'] -> ['w', 'i', 'd', 'e', 'r']


In [23]:
new_pair_counts: dict[Pair, int] = count_adjacent_pairs(merged_sequences)

new_sorted_pairs: list[tuple[Pair, int]] = sorted(
    new_pair_counts.items(),
    key=lambda item: item[1],
    reverse=True,
)

for pair, count in new_sorted_pairs:
    print(pair, count)

('lo', 'w') 5
('w', 'e') 4
('e', 'r') 4
('e', 's') 1
('s', 't') 1
('n', 'e') 1
('e', 'w') 1
('w', 'i') 1
('i', 'd') 1
('d', 'e') 1


## 10. Training BPE Merge Rules

BPE training repeatedly learns merge rules from a training corpus.

At iteration $t$, we:

1. count adjacent symbol pairs,
2. select the most frequent pair,
3. merge all non-overlapping occurrences of that pair,
4. store the selected pair as a learned merge rule.

The result is an **ordered list of merge rules**:

$$
r_1, r_2, \ldots, r_M.
$$

The order matters because later rules may depend on symbols created by
earlier rules.

For example:

<pre>
(l, o)   -> lo
(lo, w)  -> low
(e, r)   -> er
(low, er) -> lower
</pre>

The rule `(lo, w)` cannot be applied before `(l, o)` has created the
symbol `lo`.

BPE training therefore learns both:

- new vocabulary symbols,
- an ordered merge procedure.

In [24]:
def most_frequent_pair(
    pair_counts: dict[Pair, int],
) -> Pair:
    if not pair_counts:
        raise ValueError("Cannot select from empty pair counts.")

    pair, _ = min(
        pair_counts.items(),
        key=lambda item: (-item[1], item[0]),
    )

    return pair

In [27]:
def train_bpe(
    sequences: Sequence[Sequence[str]],
    num_merges: int,
) -> tuple[list[list[str]], list[Pair]]:
    current_sequences: list[list[str]] = [
        list(sequence) for sequence in sequences
    ]

    merge_rules: list[Pair] = []

    for _ in range(num_merges):
        pair_counts: dict[Pair, int] = count_adjacent_pairs(current_sequences)

        if not pair_counts:
            break

        best_pair: Pair = most_frequent_pair(pair_counts)

        current_sequences = merge_corpus(
            current_sequences,
            best_pair,
        )

        merge_rules.append(best_pair)

    return current_sequences, merge_rules


trained_sequences, merge_rules = train_bpe(
    symbol_sequences,
    num_merges=5,
)

print("merge rules:")

for index, pair in enumerate(merge_rules):
    print(
        index,
        pair,
        "->",
        pair[0] + pair[1],
    )

print("\nfinal training sequences:")

for sequence in trained_sequences:
    print(sequence)

merge rules:
0 ('l', 'o') -> lo
1 ('lo', 'w') -> low
2 ('e', 'r') -> er
3 ('low', 'er') -> lower
4 ('d', 'er') -> der

final training sequences:
['low']
['lower']
['low', 'e', 's', 't']
['low']
['lower']
['n', 'e', 'w', 'er']
['w', 'i', 'der']


## 11. BPE Training vs BPE Encoding

BPE has two different phases.

### Training

During training, pair frequencies are computed from the training corpus.

The tokenizer learns an ordered list of merge rules:

$$
r_1, r_2, \ldots, r_M.
$$

### Encoding

When encoding new text, pair frequencies are **not recomputed**.

Instead, the tokenizer starts from the base symbols and applies the
previously learned merge rules in their learned order.

Therefore:

<pre>
training corpus
      ↓
learn merge rules
      ↓
fixed tokenizer
      ↓
encode new text

In [28]:
def apply_merge_rules(
    sequence: Sequence[str], merge_rules: Sequence[Pair]
) -> list[str]:
    current: list[str] = list(sequence)

    for pair in merge_rules:
        current = merge_pair(current, pair)

    return current


# known word test
word: str = "lower"

tokens: list[str] = apply_merge_rules(
    list(word),
    merge_rules,
)

print("word:", word)
print("tokens:", tokens)


# unknown word test
for word in [
    "low",
    "lower",
    "lowest",
    "lowering",
]:
    tokens: list[str] = apply_merge_rules(
        list(word),
        merge_rules,
    )

    print(word, "->", tokens)

word: lower
tokens: ['lower']
low -> ['low']
lower -> ['lower']
lowest -> ['low', 'e', 's', 't']
lowering -> ['lower', 'i', 'n', 'g']


In [30]:
def build_bpe_vocabulary(
    base_symbols: Sequence[str], merge_rules: Sequence[Pair]
) -> list[str]:
    vocabulary: list[str] = list(base_symbols)

    for left, right in merge_rules:
        merged_symbol: str = left + right

        if merged_symbol not in vocabulary:
            vocabulary.append(merged_symbol)

    return vocabulary


base_symbols: list[str] = sorted(
    {symbol for sequence in symbol_sequences for symbol in sequence}
)

bpe_vocab: list[str] = build_bpe_vocabulary(
    base_symbols,
    merge_rules,
)

print("base vocabulary:", base_symbols)
print("BPE vocabulary:", bpe_vocab)
print("size:", len(bpe_vocab))

base vocabulary: ['d', 'e', 'i', 'l', 'n', 'o', 'r', 's', 't', 'w']
BPE vocabulary: ['d', 'e', 'i', 'l', 'n', 'o', 'r', 's', 't', 'w', 'lo', 'low', 'er', 'lower', 'der']
size: 15


In [31]:
word = "lower"

character_tokens: list[str] = list(word)

bpe_tokens: list[str] = apply_merge_rules(
    character_tokens,
    merge_rules,
)

print("character tokens:", character_tokens)
print("character length:", len(character_tokens))

print("BPE tokens:", bpe_tokens)
print("BPE length:", len(bpe_tokens))

character tokens: ['l', 'o', 'w', 'e', 'r']
character length: 5
BPE tokens: ['lower']
BPE length: 1


## Character, Subword, and Byte Tokenization

The tokenizer implemented in this lesson begins from Python characters.

This is sufficient to understand the main ideas behind vocabulary
construction and BPE, but modern tokenizers may use different base
symbols.

### Character-level tokenization

A character is treated as the smallest symbol.

For example:

<pre>
hello
↓
h | e | l | l | o
</pre>

This is simple, but sequences can become long and unseen characters may
not exist in the vocabulary.

### Subword tokenization

BPE learns larger symbols by repeatedly merging frequently occurring
adjacent symbols.

For example:

<pre>
l | o | w | e | r
↓
lo | w | e | r
↓
low | e | r
↓
low | er
↓
lower
</pre>

The learned units may represent characters, subwords, or complete words.

### Byte-level tokenization

Another option is to begin from the bytes of a text encoding such as
UTF-8 rather than directly from Unicode characters.

Because a byte can take only 256 possible values, the base vocabulary
can represent arbitrary byte sequences.

This greatly reduces the problem of encountering completely unknown
Unicode characters.

The BPE algorithm itself does not fundamentally change:

<pre>
base symbols
    ↓
count adjacent pairs
    ↓
merge frequent pairs
    ↓
larger learned symbols
</pre>

What changes is the definition of the initial symbols.

## Limitations of This Implementation

The BPE implementation in this lesson is intentionally minimal.

It demonstrates the core learning algorithm:

- initialize a sequence of symbols,
- count adjacent symbol pairs,
- select a frequent pair,
- merge it,
- store the merge rule,
- apply learned rules during encoding.

It does not attempt to reproduce a production tokenizer.

In particular, it does not currently implement:

- byte-level preprocessing,
- Unicode normalization,
- sophisticated pre-tokenization,
- special-token handling,
- tokenizer serialization,
- optimized pair-frequency updates,
- large-corpus training.

These features are important in practical tokenizer systems, but they
are separate from the central BPE mechanism studied here.

For later language-model experiments, we can use either this educational
tokenizer or a mature tokenizer implementation depending on the purpose
of the experiment.

## 12. Byte-Level Tokenization

A character-level tokenizer uses Unicode characters as its base symbols.

A byte-level tokenizer instead begins from the bytes produced by an
encoding such as UTF-8.

UTF-8 represents text using byte values in the range

$$
0, 1, \ldots, 255.
$$

This gives byte-level tokenization an important property:

> Any valid UTF-8 text can be represented using a fixed set of at most
> 256 base byte values.

Different Unicode characters may require different numbers of bytes.

For example, an ASCII character usually occupies one UTF-8 byte, while
many accented characters, Chinese characters, and emoji require
multiple bytes.

BPE can then operate on these byte symbols in exactly the same conceptual
way as before:

<pre>
bytes
  ↓
count adjacent symbol pairs
  ↓
merge frequent pairs
  ↓
larger byte sequences
</pre>

The BPE algorithm does not fundamentally change. Only the initial symbol
representation changes.

In [34]:
def inspect_utf8(text: str) -> None:
    encoded: bytes = text.encode("utf-8")

    print("text:", repr(text))
    print("bytes:", encoded)
    print("byte values:", list(encoded))
    print("number of characters:", len(text))
    print("number of bytes:", len(encoded))


inspect_utf8("A")
inspect_utf8("é")
inspect_utf8("你")
inspect_utf8("🙂")

text: 'A'
bytes: b'A'
byte values: [65]
number of characters: 1
number of bytes: 1
text: 'é'
bytes: b'\xc3\xa9'
byte values: [195, 169]
number of characters: 1
number of bytes: 2
text: '你'
bytes: b'\xe4\xbd\xa0'
byte values: [228, 189, 160]
number of characters: 1
number of bytes: 3
text: '🙂'
bytes: b'\xf0\x9f\x99\x82'
byte values: [240, 159, 153, 130]
number of characters: 1
number of bytes: 4


In [35]:
ByteSymbol = bytes
BytePair = tuple[ByteSymbol, ByteSymbol]
ByteSequence = list[ByteSymbol]

In [37]:
def text_to_byte_symbols(text: str) -> ByteSequence:
    return [bytes([byte_value]) for byte_value in text.encode("utf-8")]


symbols: ByteSequence = text_to_byte_symbols("你a")

print(symbols)

[b'\xe4', b'\xbd', b'\xa0', b'a']


In [38]:
def byte_symbols_to_text(symbols: ByteSequence) -> str:
    byte_stream: bytes = b"".join(symbols)
    return byte_stream.decode("utf-8")


text: str = "hello 你好 🙂"

symbols: ByteSequence = text_to_byte_symbols(text)
decoded: str = byte_symbols_to_text(symbols)

print(repr(text))
print(symbols)
print(repr(decoded))

assert decoded == text

'hello 你好 🙂'
[b'h', b'e', b'l', b'l', b'o', b' ', b'\xe4', b'\xbd', b'\xa0', b'\xe5', b'\xa5', b'\xbd', b' ', b'\xf0', b'\x9f', b'\x99', b'\x82']
'hello 你好 🙂'


In [39]:
def merge_byte_pair(
    sequence: ByteSequence,
    pair: BytePair,
) -> ByteSequence:
    merged: ByteSequence = []
    index: int = 0

    while index < len(sequence):
        if (
            index < len(sequence) - 1
            and sequence[index] == pair[0]
            and sequence[index + 1] == pair[1]
        ):
            merged.append(pair[0] + pair[1])
            index += 2
        else:
            merged.append(sequence[index])
            index += 1

    return merged

## 13. Unicode Normalization

Unicode allows some visually identical text to be represented by
different sequences of code points.

For example, an accented character may be represented either as a
single precomposed code point or as a base character followed by a
combining mark.

These strings may look identical to a human reader while being different
sequences for a tokenizer.

Unicode normalization defines standard transformations between such
representations.

Two common forms are:

- **NFC**: canonical composition,
- **NFD**: canonical decomposition.

Normalization is a tokenizer design decision because it changes the
symbol sequence seen during vocabulary learning and encoding.

In [ ]:
# example
import unicodedata


composed: str = "é"
decomposed: str = "e\u0301"

print(composed)
print(decomposed)

print("equal:", composed == decomposed)
print("composed length:", len(composed))
print("decomposed length:", len(decomposed))

é
é
equal: False
composed length: 1
decomposed length: 2


In [41]:
def code_points(text: str) -> list[str]:
    return [f"U+{ord(char):04X}" for char in text]


print(code_points(composed))
print(code_points(decomposed))

['U+00E9']
['U+0065', 'U+0301']


In [43]:
# Normalization
composed_nfc: str = unicodedata.normalize(
    "NFC",
    decomposed,
)

print(composed_nfc == composed)


decomposed_nfd: str = unicodedata.normalize(
    "NFD",
    composed,
)

print(decomposed_nfd == decomposed)

True
True


## 14. Special Tokens

Not every token represents ordinary text.

Language models often use reserved tokens to represent control
information such as sequence boundaries or padding.

Examples include:

- `<bos>` — beginning of sequence,
- `<eos>` — end of sequence,
- `<pad>` — padding,
- `<unk>` — unknown token.

These tokens are part of the tokenizer/model protocol rather than
ordinary text discovered through BPE merges.

A special token therefore has two roles:

1. a symbolic meaning,
2. a reserved token ID understood by both the tokenizer and the model.

Special tokens should not accidentally collide with normal vocabulary
entries.

In [44]:
# example
vocab: list[str] = [
    "a",
    "b",
    "c",
]

token_to_id: dict[str, int] = {
    token: index for index, token in enumerate(vocab)
}


special_tokens: tuple[str, ...] = (
    "<bos>",
    "<eos>",
    "<pad>",
)

next_id: int = len(token_to_id)

for token in special_tokens:
    token_to_id[token] = next_id
    next_id += 1

## Final Tokenization Model

A practical tokenizer can be understood as several layers:

<pre>
raw Unicode text
       ↓
optional normalization
       ↓
base representation
(character or byte symbols)
       ↓
learned BPE merge rules
       ↓
subword tokens
       ↓
token IDs
       ↓
optional special-token insertion
       ↓
model input sequence
</pre>

These steps solve different problems.

Unicode normalization controls text representation.

Byte-level tokenization provides broad input coverage.

BPE learns a compressed reusable subword vocabulary.

Special tokens provide a control protocol between the tokenizer and the
language model.